In [12]:
import awswrangler as awr
import pandas as pd
import os
import openpyxl

### GERANDO dataframes

In [ ]:
#gerando todos os boletos pagos
caminho_query = r"C:\Users\raphael.almeida\Documents\Processos\C:\Users\raphael.almeida\Documents\Processos\relatorio_placas_adimplentes\sql\faturas_baixadas.sql"
with open(caminho_query,'r') as arquivo_query:
    query = arquivo_query.read()
df_adimplentes = awr.athena.read_sql_query(query, database='silver')

In [ ]:
#gerando todos os boletos posvencimento_recente
caminho_query = r"C:\Users\raphael.almeida\Documents\Processos\C:\Users\raphael.almeida\Documents\Processos\relatorio_placas_adimplentes\sql\faturas_posvencimento_recente.sql"
with open(caminho_query,'r') as arquivo_query:
    query = arquivo_query.read()
df_faturas_recente = awr.athena.read_sql_query(query, database='silver')

In [ ]:
#gerando todos os boletos posvencimento_45
caminho_query = r"C:\Users\raphael.almeida\Documents\Processos\C:\Users\raphael.almeida\Documents\Processos\relatorio_placas_adimplentes\sql\faturas_posvencimento_45.sql"
with open(caminho_query,'r') as arquivo_query:
    query = arquivo_query.read()
df_faturas_45 = awr.athena.read_sql_query(query, database='silver')

### MODULANDO dataframes (com as categorias necessárias e concatenando)

In [16]:
#gerando df_adimplentes
boletos_pagos = set(
    zip(
        df_adimplentes['ponteiro'],
        df_adimplentes['empresa'],
        df_adimplentes['conjunto']
    )
)
boletos_pagos

{(np.int64(2559991), 'Tag', np.int64(2681)),
 (np.int64(2591751), 'Tag', np.int64(5356)),
 (np.int64(2576655), 'Tag', np.int64(4156)),
 (np.int64(2673535), 'Tag', np.int64(11431)),
 (np.int64(464372), 'Viavante', np.int64(18770)),
 (np.int64(2550535), 'Tag', np.int64(1916)),
 (np.int64(2542660), 'Tag', np.int64(1088)),
 (np.int64(264573), 'Stcoop', np.int64(16553)),
 (np.int64(402239), 'Viavante', np.int64(16800)),
 (np.int64(2681966), 'Tag', np.int64(12027)),
 (np.int64(2642781), 'Tag', np.int64(9197)),
 (np.int64(256388), 'Stcoop', np.int64(16049)),
 (np.int64(2618944), 'Tag', np.int64(7557)),
 (np.int64(438291), 'Viavante', np.int64(19309)),
 (np.int64(2595069), 'Tag', np.int64(5741)),
 (np.int64(2565627), 'Tag', np.int64(3269)),
 (np.int64(2676980), 'Tag', np.int64(11664)),
 (np.int64(417840), 'Viavante', np.int64(17802)),
 (np.int64(410797), 'Viavante', np.int64(17448)),
 (np.int64(2578936), 'Tag', np.int64(4445)),
 (np.int64(404596), 'Viavante', np.int64(17004)),
 (np.int64(43657

In [17]:
#gerando df_inadimplentes_recente
df_inadimplentes_recente = df_faturas_recente[
    ~df_faturas_recente.apply(
        lambda row: (row['ponteiro'], row['empresa'], row['conjunto']) in boletos_pagos, axis=1
    )
]

In [18]:
#gerando df_inadimplentes_45
df_inadimplentes_45 = df_faturas_45[
    ~df_faturas_45.apply(
        lambda row: (row['ponteiro'], row['empresa'], row['conjunto']) in boletos_pagos, axis=1
    )
]

In [19]:
# a pedido do setor de COBRANÇAS, retirar associado: APROSSIL - ASSOCIACAO DE PROPRIETARIOS DE CAMINHOES DO SUL D
# a pedido do setor de COBRANÇAS, retirar associado: TESTE
df_inadimplentes_recente = df_inadimplentes_recente[
    ~(
        (df_inadimplentes_recente['empresa'].isin(['Viavante', 'Stcoop', 'Segtruck'])) &
        (df_inadimplentes_recente['associado'] == "APROSSIL - ASSOCIACAO DE PROPRIETARIOS DE CAMINHOES DO SUL D")
    )
]
df_inadimplentes_recente = df_inadimplentes_recente[~df_inadimplentes_recente['associado'].str.contains('TESTE', na=False)]

In [20]:
# a pedido do setor de COBRANÇAS, retirar associado: APROSSIL - ASSOCIACAO DE PROPRIETARIOS DE CAMINHOES DO SUL D
# a pedido do setor de COBRANÇAS, retirar associado: TESTE
df_inadimplentes_45 = df_inadimplentes_45[
    ~(
        (df_inadimplentes_45['empresa'].isin(['Viavante', 'Stcoop', 'Segtruck'])) &
        (df_inadimplentes_45['associado'] == "APROSSIL - ASSOCIACAO DE PROPRIETARIOS DE CAMINHOES DO SUL D")
    )
]
df_inadimplentes_45 = df_inadimplentes_45[~df_inadimplentes_45['associado'].str.contains('TESTE', na=False)]

In [21]:
df_inadimplentes_recente["pagamento"] = "inadimplente"
df_inadimplentes_45["pagamento"] = "inadimplente 45+"
df_adimplentes["pagamento"] = "adimplente"
df_placas_pagamento = pd.concat([df_inadimplentes_recente, df_inadimplentes_45, df_adimplentes], ignore_index=True)

In [22]:
caminho_pasta = r'C:\Users\raphael.almeida\OneDrive - Grupo Unus\analise de dados - Arquivos em excel\Relatório Adimplência e Inadimplência'
caminho_arquivo = os.path.join(caminho_pasta, 'relatorio_adimplencia_inadimplencia.xlsx')
os.makedirs(caminho_pasta, exist_ok=True)
if os.path.exists(caminho_arquivo):
    os.remove(caminho_arquivo)
    print("Arquivo antigo removido, iniciando carregamento...")
df_placas_pagamento.to_excel(caminho_arquivo, engine='openpyxl', index=False, sheet_name='faturas')
print(f"Arquivo Excel salvo com sucesso em: {caminho_arquivo}")

Arquivo antigo removido, iniciando carregamento...
Arquivo Excel salvo com sucesso em: C:\Users\raphael.almeida\OneDrive - Grupo Unus\analise de dados - Arquivos em excel\Relatório Adimplência e Inadimplência\relatorio_adimplencia_inadimplencia.xlsx
